In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.appName('Local Batch') \
    .config("spark.mongodb.output.uri", "mongodb://127.0.0.1:27017/crypto_database.hourly_stats") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.13:10.2.1") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/12/29 11:46:37 WARN Utils: Your hostname, tomek-arch, resolves to a loopback address: 127.0.1.1; using 192.168.1.124 instead (on interface wlp1s0)
25/12/29 11:46:37 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/tomek2/studia/semestr_7/bigdata/projekt/crypto/venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/tomek2/.ivy2.5.2/cache
The jars for the packages stored in: /home/tomek2/.ivy2.5.2/jars
org.mongodb.spark#mongo-spark-connector_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-4479155b-7691-41b1-a78e-0b0a79dd3a2a;1.0
	confs: [default]
	found org.mongodb.spark#mongo-spark-connector_2.13;10.2.1 in central
	found org.mongodb#mongodb-driver-sync;4.8.2 in central
	[4.8.2] org.mongodb#mongodb-driver-sync;[4.8

In [3]:
path = '../../data/prices'
df = spark.read.parquet(path)

In [4]:
df.show(3)

+--------+------------------+-------------------+----+-----+---+
|currency|             price|          timestamp|year|month|day|
+--------+------------------+-------------------+----+-----+---+
| bitcoin| 86782.69279278311|2025-12-16 02:07:04|2025|   12| 16|
|ethereum| 2584.649518704182|2025-12-16 02:07:04|2025|   12| 16|
|dogecoin|0.1981388398077129|2025-12-16 02:07:04|2025|   12| 16|
+--------+------------------+-------------------+----+-----+---+
only showing top 3 rows


## Cogodzinne agregacje

In [16]:
from datetime import datetime

from pyspark.sql.functions import col, hour, std, mean
from pyspark.sql.window import Window
import pyspark.sql.functions as F

# target_time = datetime.now()
# target_year = target_time.year
# target_month = target_time.month
# target_day = target_time.day
# target_hour = target_time.hour

target_year = 2025
target_month = 12
target_day = 16
target_hour = 1



In [17]:
# Read i od razu filtrowanie po godzinie

df = spark.read.parquet(path) \
    .filter((col("year") == target_year) &
            (col("month") == target_month) &
            (col("day") == target_day) &
            (hour(col("timestamp")) == target_hour))

window = Window.partitionBy("currency","year", "month", "day", "hour").orderBy("timestamp")

df_enriched = df.withColumn("hour", hour(col("timestamp"))) \
                .withColumn("open_price", F.first("price").over(window)) \
                .withColumn("close_price", F.last("price").over(window))

# 1. Najprostsze statystystyki godzinowe

final_batch_view = df_enriched.groupBy("currency", "year", "month", "day", "hour") \
                              .agg(F.first("open_price").alias("open"),
                                   F.max("price").alias("high"),
                                   F.min("price").alias("low"),
                                   F.last("close_price").alias("close"),
                                   F.count("price").alias("read_count"),
                                   mean(col("price")).alias("mean_price"), 
                                   std(col("price")).alias("std_price"))

final_batch_view = final_batch_view.withColumn("return_pct", (col("close") - col("open")) / col("open") * 100) \
                                   .withColumn("spread_pct", (col("high") - col("low")) / col("low") * 100)

In [15]:
final_batch_view.show()

+--------+----+-----+---+----+-------------------+-------------------+-------------------+-------------------+----------+------------------+--------------------+-------------------+------------------+
|currency|year|month|day|hour|               open|               high|                low|              close|read_count|        mean_price|           std_price|         return_pct|        spread_pct|
+--------+----+-----+---+----+-------------------+-------------------+-------------------+-------------------+----------+------------------+--------------------+-------------------+------------------+
| bitcoin|2025|   12| 16|   2|  93142.65663587954| 108184.85551353998|  74118.39770496625|  90131.51853022238|       719| 90014.16258052808|   4848.861374713642|-3.2328239438440423| 45.96221567575945|
|dogecoin|2025|   12| 16|   2|0.20251521418800575|0.21694818422313467|0.18209220892987843|0.19300638893396535|       719|0.2000516753343755|0.005274092597569065| -4.695363403765232| 19.14193665840

In [12]:
final_batch_view.write.format("mongo").mode("append").save()

Py4JJavaError: An error occurred while calling o300.save.
: org.apache.spark.SparkClassNotFoundException: [DATA_SOURCE_NOT_FOUND] Failed to find the data source: mongo. Make sure the provider name is correct and the package is properly registered and compatible with your Spark version. SQLSTATE: 42K02
	at org.apache.spark.sql.errors.QueryExecutionErrors$.dataSourceNotFoundError(QueryExecutionErrors.scala:722)
	at org.apache.spark.sql.execution.datasources.DataSource$.lookupDataSource(DataSource.scala:681)
	at org.apache.spark.sql.execution.datasources.DataSource$.lookupDataSourceV2(DataSource.scala:740)
	at org.apache.spark.sql.classic.DataFrameWriter.lookupV2Provider(DataFrameWriter.scala:626)
	at org.apache.spark.sql.classic.DataFrameWriter.saveInternal(DataFrameWriter.scala:135)
	at org.apache.spark.sql.classic.DataFrameWriter.save(DataFrameWriter.scala:126)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)
	at java.base/jdk.internal.reflect.NativeMe